In [1]:
import torch
import pandas as pd
import time
from tqdm import tqdm
from unsloth import FastLanguageModel

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("PyTorch version:", torch.__version__)
print("Torch Built With CUDA:", torch.backends.cuda.is_built())
print("Current CUDA Device:", torch.cuda.current_device())
print("CUDA Version (Torch):", torch.version.cuda)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/sdonev/data/conda/envs/unsloth_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Unsloth: Failed to patch SmolVLMForConditionalGeneration forward function.
🦥 Unsloth Zoo will now patch everything to make training faster!
CUDA available: True
GPU: NVIDIA H100 80GB HBM3
PyTorch version: 2.5.1
Torch Built With CUDA: True
Current CUDA Device: 0
CUDA Version (Torch): 12.1


# Load Model

In [2]:
model_name="unsloth/Meta-Llama-3.1-8B"
model_name_str = model_name.lower().replace("-","_").replace("/","_")
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.


In [3]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=f"/data/sdonev/llms/lora_model_{model_name_str}",        # points to the folder with saved model
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2025.4.1: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.209 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1. CUDA: 9.0. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.4.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [4]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

In [5]:
inputs = tokenizer(
[
    alpaca_prompt.format(
        "What is the age of the used animals in this experiments? Return `AGE: <standardized age or descriptor>`", # instruction
        "Mice used in these studies were 2-3 month old males with a mixed C57BL/6 6 129S background", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs)

['<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nWhat is the age of the used animals in this experiments? Return `AGE: <standardized age or descriptor>`\n\n### Input:\nMice used in these studies were 2-3 month old males with a mixed C57BL/6 6 129S background\n\n### Response:\nAGE: 2-3 months<|end_of_text|>']

# Load Data

In [7]:
bert_classified = pd.read_csv("./model_predictions/age/sentences_with_bert_age_predictions_20250721.csv")
bert_classified['doc_id_unique'] = bert_classified['PMID'].astype(str) + "_" + bert_classified['sentence_id'].astype(str).str.strip()

bert_classified.head()

,PMID,sentence_id,sent_txt,predicted_label,doc_id_unique
0,38344952,15,Encapsulation Efficiency and Drug Release The ...,0,38344952_15
1,38344952,70,Rotenone and melatonin were administered at 2....,0,38344952_70
2,15788759,0,Materials and Methods Animal treatment groups....,1,15788759_0
3,15788759,1,"Mice were fed control chow (0.09% DHA;n= 8), l...",0,15788759_1
4,11152731,0,Materials and Methods Hippocampal slices (300 ...,0,11152731_0


In [8]:
bert_classified.predicted_label.value_counts()

predicted_label
0    107611
1     21005
Name: count, dtype: int64

In [9]:
bert_classified_pos = bert_classified[bert_classified["predicted_label"]==1]
bert_classified_pos.shape

(21005, 5)

# Inference

In [10]:
instructions_simpler = """### TASK ###

EXTRACT THE AGE OF ANIMALS MENTIONED IN THE TEXT.  
RETURN ONLY THE AGE IN A STANDARDIZED FORMAT.  
IF NO AGE IS GIVEN, RETURN `"AGE NOT SPECIFIED"`.

---

### HOW TO THINK (CHAIN OF THOUGHTS) ###

1. READ the sentence carefully.
2. FIND any age-related phrases (e.g., "8 weeks", "P30", "adult", "3 months").
3. LINK each age phrase to the animal(s) it describes.
4. STANDARDIZE the format:  
   - Use `<number> <unit>` (e.g., `8 weeks`, `3 months`)  
   - Keep terms like `adult`, `juvenile`, `neonatal` unchanged  
5. IF no age is mentioned, write: `"AGE NOT SPECIFIED"`

---

### OUTPUT FORMAT ###

- `AGE: <standardized age or descriptor>`

---

### EXAMPLES ###

#### INPUT 1 ####  
Gene deletion was induced in male and female 12- to 20-week-old mice.  
#### OUTPUT 1 ####  
AGE: 12-20 weeks

#### INPUT 2 ####  
Six adult male WAG/Rij rats were used.  
#### OUTPUT 2 ####  
AGE: adult

#### INPUT 3 ####  
Juvenile pigs (approximately 3 months old) were used.  
#### OUTPUT 3 ####  
AGE: 3 months

#### INPUT 4 ####  
For Experiment 2, male young (3-months-old) and aged (23-months-old) rats were used.  
#### OUTPUT 4 ####  
AGE: 3 months, 23 months

#### INPUT 5 ####  
Twenty Sprague Dawley rats were used; no details were provided on age.  
#### OUTPUT 4 ####  
AGE: AGE NOT SPECIFIED

---

### WHAT NOT TO DO ###

- DO NOT include the whole sentence in the output  
- DO NOT include weight, sex, or strain  
- DO NOT guess the age  
- DO NOT omit the unit (e.g., weeks/months)  
- DO NOT ignore terms like "adult", "neonatal", or "juvenile"  
- DO NOT return multiple values or unformatted strings

---
ONLY OUTPUT THE AGE USING THE FORMAT ABOVE. NOTHING ELSE.
"""

In [11]:
inputs = tokenizer(
[
    alpaca_prompt.format(
        instructions_simpler,        
        "Mice used in these studies were 2-3 month old males with a mixed C57BL/6 6 129S background", # instruction
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 500, use_cache = True)
tokenizer.batch_decode(outputs)

['<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\n### TASK ###\n\nEXTRACT THE AGE OF ANIMALS MENTIONED IN THE TEXT.  \nRETURN ONLY THE AGE IN A STANDARDIZED FORMAT.  \nIF NO AGE IS GIVEN, RETURN `"AGE NOT SPECIFIED"`.\n\n---\n\n### HOW TO THINK (CHAIN OF THOUGHTS) ###\n\n1. READ the sentence carefully.\n2. FIND any age-related phrases (e.g., "8 weeks", "P30", "adult", "3 months").\n3. LINK each age phrase to the animal(s) it describes.\n4. STANDARDIZE the format:  \n   - Use `<number> <unit>` (e.g., `8 weeks`, `3 months`)  \n   - Keep terms like `adult`, `juvenile`, `neonatal` unchanged  \n5. IF no age is mentioned, write: `"AGE NOT SPECIFIED"`\n\n---\n\n### OUTPUT FORMAT ###\n\n- `AGE: <standardized age or descriptor>`\n\n---\n\n### EXAMPLES ###\n\n#### INPUT 1 ####  \nGene deletion was induced in male and female 12- to 20-week-old mice.  

In [12]:
def extract_animal_age_unsloth(text, model, tokenizer, instructions):
    inputs = tokenizer(
        [
            alpaca_prompt.format(
                instructions,        
                text, # instruction
                "", # output - leave this blank for generation!
            )
        ], return_tensors = "pt").to("cuda")
        
    outputs = model.generate(**inputs, max_new_tokens = 500, use_cache = True)
    res = tokenizer.batch_decode(outputs)
    return res[0].split("Response:")[1].replace("<|end_of_text|>","").strip()

In [101]:
import re
from num2words import num2words
import logging
from datetime import datetime


In [102]:

def is_valid_age_prediction(prediction: str, text: str) -> bool:
    if not prediction:
        return False

    standalone_labels = {
        "young adult", "adult", "juvenile", "neonatal",
        "aged", "young", "old", "newborn"
    }

    entries = [entry.strip() for entry in prediction.split(",") if entry.strip()]
    if not entries:
        return False

    text_lower = text.lower()

    for entry in entries:
        entry = entry.lower()
        words = entry.split()

        # Allow single-word predictions like "adult"
        if len(words) == 1 and words[0] in standalone_labels:
            continue

        if len(words) != 2:
            return False

        number_part, unit = words

        if not re.fullmatch(r"(day|days|week|weeks|month|months|year|years)", unit, re.IGNORECASE):
            return False

        # Handle range like "17–19 weeks"
        range_match = re.match(r"(\d+)[–-](\d+)", number_part)
        if range_match:
            number = range_match.group(2)  # Take the second number
        else:
            number = number_part

        # Try both digit and word forms
        digit_form = number
        try:
            word_form = num2words(int(float(number)))  # "8" -> "eight"
        except:
            word_form = number  # fallback

        if (digit_form not in text_lower) and (word_form not in text_lower):
            return False

    return True

In [103]:
is_valid_age_prediction("adult (810 weeks) or aged (80 weeks)","the study was carried out on adult (810 weeks) or aged (80 weeks) mice")

False

In [104]:
is_valid_age_prediction("8 weeks"," ten-week-old female C57BL/6 mice were acquired from Orient Bio Inc")

False

In [105]:

def run_predictions_and_collect_unsloth(
    df,
    text_col,
    model,
    tokenizer,
    instructions,
    output_path=None,
    max_attempts=3,
    log_path="server_logs/llm_age_prediction.log"
):
    # Setup logger
    logging.basicConfig(
        filename=log_path,
        filemode='w',  # Overwrite log on each run; use 'a' to append
        level=logging.INFO,
        format="%(asctime)s - %(message)s",
    )
    
    predictions = []

    for i, row in tqdm(df.iterrows(), total=len(df)):
        pmid = row["PMID"]
        doc_id = row["doc_id_unique"]
        text = row[text_col]

        try:
            attempted_predictions = []
            for attempt in range(1, max_attempts + 1):
                prediction = extract_animal_age_unsloth(text, model, tokenizer, instructions)
                cleaned_prediction = prediction.replace("AGE:", "").strip()
                attempted_predictions.append(prediction)

                if is_valid_age_prediction(cleaned_prediction, text):
                    age_prediction = prediction
                    break
                logging.info(f"[{pmid}][Attempt {attempt}] Invalid prediction: {prediction} for text: {text}")
            else:
                age_prediction = max(attempted_predictions, key=lambda p: len(p)) if attempted_predictions else "ERROR: No valid attempts"

        except Exception as e:
            age_prediction = f"ERROR: {str(e)}"

        logging.info(f"[{pmid}][Final prediction]: {age_prediction}")

        prediction_row = {
            "pmid": pmid,
            "doc_id_unique": doc_id,
            "ent_text": text,
            "age_prediction": age_prediction
        }

        predictions.append(prediction_row)

        if output_path:
            pd.DataFrame([prediction_row]).to_csv(
                output_path, mode='a', header=(i == 0), index=False
            )

    return pd.DataFrame(predictions)


In [106]:

# Generate a timestamp like '20250721_1530'
timestamp = datetime.now().strftime("%Y%m%d")
timestamp

'20250721'

In [107]:
f"predictions/age_{model_name_str}.csv"

'predictions/age_unsloth_meta_llama_3.1_8b.csv'

In [ ]:
start_time = time.time()

predictions = run_predictions_and_collect_unsloth(
    bert_classified_pos,
    "sent_txt",
    model,
    tokenizer,
    instructions_simpler, #prompt_full_sentences, #instructions_simpler,
    output_path=f"model_predictions/age/age_{model_name_str}_{timestamp}.csv"
)

end_time = time.time()
elapsed = end_time - start_time
print(f"\nCompleted predictions in {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")

  2%|▏         | 489/21005 [01:50<1:29:16,  3.83it/s]